# 🧹 Module 7 · Data Cleaning and Transformation
### Course: Python from Scratch for Data Analysis

**Prerequisites:** Modules 1–6

### What you will learn today
1. Detecting and handling missing values
2. Fixing data types and cleaning strings
3. Groupby and aggregation (in depth)
4. End-to-end cleaning mini-project

## The dataset we will work with today

This is a **deliberately messy** employee dataset. It has missing values, wrong data types, inconsistent capitalization, and extra spaces. Our job in this module is to fix every one of these problems — step by step.

In [1]:
import pandas as pd
import numpy as np

# This dataset intentionally has problems — we will fix them one by one.
raw_data = {
    "name":       ["Valentina Torres", "SANTIAGO GÓMEZ", "camila restrepo", "Andrés Martínez",
                   "Isabella Vargas", "daniel herrera", "Sofía López", None,
                   "Mariana Díaz", "Julián Pérez", "  Laura Suárez  ", "Carlos Rodríguez"],
    "department": ["Engineering", "marketing", "Engineering", "HR", "Engineering",
                   "Marketing", "hr", "Engineering", "Marketing", "HR", "Engineering", "Marketing"],
    "salary":     [4500000, "3200000", 5100000, None, 4800000,
                   3500000, 3100000, 5500000, None, 2800000, 4200000, 3700000],
    "years_exp":  [5, 3, 7, 2, None, 4, 1, 9, 3, 2, 5, None],
    "city":       ["Bogotá", "Medellín", "Bogotá", "Cali ", " Bogotá", "Medellín",
                   "Cali", "bogotá", "Medellín", "Cali", "Bogotá", "Medellín"],
    "start_date": ["2019-03-15", "2021-06-01", "2017-08-20", "2022-01-10",
                   "2018-05-07", "2020-11-30", "2023-02-14", "2015-09-01",
                   "2021-07-22", "2022-03-05", "2019-12-01", "2020-04-18"]
}
df_raw = pd.DataFrame(raw_data)
print("Dataset loaded. Shape:", df_raw.shape)
df_raw

Dataset loaded. Shape: (12, 6)


,name,department,salary,years_exp,city,start_date
0,Valentina Torres,Engineering,4500000,5.0,Bogotá,2019-03-15
1,SANTIAGO GÓMEZ,marketing,3200000,3.0,Medellín,2021-06-01
2,camila restrepo,Engineering,5100000,7.0,Bogotá,2017-08-20
3,Andrés Martínez,HR,None,2.0,Cali,2022-01-10
4,Isabella Vargas,Engineering,4800000,NaN,Bogotá,2018-05-07
5,daniel herrera,Marketing,3500000,4.0,Medellín,2020-11-30
6,Sofía López,hr,3100000,1.0,Cali,2023-02-14
7,NaN,Engineering,5500000,9.0,bogotá,2015-09-01
8,Mariana Díaz,Marketing,None,3.0,Medellín,2021-07-22
9,Julián Pérez,HR,2800000,2.0,Cali,2022-03-05


---
## 🔷 Block 1 — Detecting and Handling Missing Values

### Analogy: the blank form

Imagine you are reviewing paper registration forms submitted by employees. Some people left the salary field blank. Some did not write their city. What do you do?

You have three choices:
1. **Ask again** — go back and collect the missing data (not always possible).
2. **Use an estimate** — fill in a reasonable value, like the department average.
3. **Skip that form** — if too much is missing, the row is not useful.

Pandas gives you tools to make exactly these decisions. A missing value in Pandas is called **NaN** (Not a Number) — it is a special marker that means "we don't know this value."

### Step 1 — Detect missing values

In [2]:
# .isna() returns a boolean DataFrame: True where a value is missing.
df_raw.isna()

,name,department,salary,years_exp,city,start_date
0,False,False,False,False,False,False
1,False,False,False,False,False,False
2,False,False,False,False,False,False
3,False,False,True,False,False,False
4,False,False,False,True,False,False
5,False,False,False,False,False,False
6,False,False,False,False,False,False
7,True,False,False,False,False,False
8,False,False,True,False,False,False
9,False,False,False,False,False,False


In [3]:
# The most useful command: count missing values per column.
df_raw.isna().sum()

name          1
department    0
salary        2
years_exp     2
city          0
start_date    0
dtype: int64

In [4]:
# .info() shows dtypes AND non-null counts at the same time.
df_raw.info()

<class 'pandas.DataFrame'>
RangeIndex: 12 entries, 0 to 11
Data columns (total 6 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   name        11 non-null     str    
 1   department  12 non-null     str    
 2   salary      10 non-null     object 
 3   years_exp   10 non-null     float64
 4   city        12 non-null     str    
 5   start_date  12 non-null     str    
dtypes: float64(1), object(1), str(4)
memory usage: 708.0+ bytes


**Reading the output of `.info()`:**

- Column `name`: 11 non-null — so 1 is missing.
- Column `salary`: 10 non-null — so 2 are missing. And the dtype is `object` (string), not `int64`! That is a type problem we will fix in Block 2.
- Column `years_exp`: 10 non-null — 2 missing.

`.notna()` is the opposite of `.isna()` — it returns `True` where a value **is** present.

In [5]:
# .notna() counts how many values are PRESENT (not missing).
df_raw.notna().sum()

name          11
department    12
salary        10
years_exp     10
city          12
start_date    12
dtype: int64

### Step 2 — Handle missing values: three strategies

| Situation | Strategy |
|-----------|----------|
| Row is too incomplete to be useful | `.dropna()` — remove the row |
| Numeric column, random missingness | `.fillna(mean or median)` — fill with estimate |
| Categorical column | `.fillna("Unknown")` — fill with a placeholder |
| Time-series data | `.fillna(method="ffill")` — carry the previous value forward |

In [6]:
# Strategy A: drop any row that has at least one NaN.
df_dropped_any = df_raw.dropna()
print("Original rows:", len(df_raw))
print("After dropna():", len(df_dropped_any))

Original rows: 12
After dropna(): 7


In [7]:
# Strategy B: drop only rows where a SPECIFIC column is missing.
# Here we keep rows even if years_exp is missing — but we remove rows where name is missing.
df_dropped_name = df_raw.dropna(subset=["name"])
print("After dropna(subset=['name']):", len(df_dropped_name))

After dropna(subset=['name']): 11


In [8]:
# Strategy C: drop only rows where ALL columns are NaN.
# (None of our rows are fully empty, so nothing gets dropped here.)
df_dropped_all = df_raw.dropna(how="all")
print("After dropna(how='all'):", len(df_dropped_all))

After dropna(how='all'): 12


In [9]:
# Strategy D: fill missing values with a fixed value.
# Good for categorical columns where 'Unknown' makes sense.
df_raw["name"].fillna("Unknown")

0     Valentina Torres
1       SANTIAGO GÓMEZ
2      camila restrepo
3      Andrés Martínez
4      Isabella Vargas
5       daniel herrera
6          Sofía López
7              Unknown
8         Mariana Díaz
9         Julián Pérez
10      Laura Suárez  
11    Carlos Rodríguez
Name: name, dtype: str

In [10]:
# Strategy E: fill missing numeric values with the column mean.
# Note: salary is stored as object (string) right now — we will fix the type in Block 2.
# For now, let's use years_exp which IS numeric.
median_exp = df_raw["years_exp"].median()
print("Median years_exp:", median_exp)

df_raw["years_exp"].fillna(median_exp)

Median years_exp: 3.5


0     5.0
1     3.0
2     7.0
3     2.0
4     3.5
5     4.0
6     1.0
7     9.0
8     3.0
9     2.0
10    5.0
11    3.5
Name: years_exp, dtype: float64

### Deliberate error — what happens when you apply `.mean()` to a string column?

> **Before running:** predict — will this work?

In [11]:
# Deliberate error: .mean() does not work on a string column.
# Read the error message carefully before fixing.
df_raw["name"].fillna(df_raw["name"].mean())

TypeError: Cannot perform reduction 'mean' with string dtype

**Why does this fail?** `mean()` only makes sense for numbers. The `name` column contains strings — the average of "Valentina" and "Santiago" is undefined. The fix is to use `.fillna("Unknown")` instead.

---
### Exercise 1

**🧠 Before you code — apply the 5 steps:**

- **Input:** `df_raw` — a DataFrame with missing values in `name`, `salary`, and `years_exp`.
- **Output:** A cleaned DataFrame where:
  - Rows with missing `name` are removed.
  - Missing `salary` rows are removed.
  - Missing `years_exp` is filled with the median.

**Decompose:**
1. Count the missing values in each column to confirm the problem.
2. Drop rows where `name` is missing.
3. Drop rows where `salary` is missing.
4. Calculate the median of `years_exp`, then fill the missing ones with that value.
5. Confirm no missing values remain.

**Pseudocode:**
```
Print count of missing values per column.
Remove rows where name is empty → save as cleaned DataFrame.
Remove rows from cleaned DataFrame where salary is empty.
Calculate the median of years_exp.
Fill any remaining empty years_exp with the median.
Print the missing value count again to confirm zero.
```

**Pattern:** Filter (detect and remove) + fill estimate for remaining gaps.

In [12]:
# Exercise 1 — Handle missing values in df_raw.

# Step 1: confirm the missing value counts.
print("Missing before:")
print(df_raw.isna().sum())
print()

# Step 2: drop rows where name is missing.
df_clean = df_raw.dropna(subset=["name"])

# Step 3: drop rows where salary is missing.
df_clean = df_clean.dropna(subset=["salary"])

# Step 4: fill missing years_exp with the median.
median_exp = df_clean["years_exp"].median()
df_clean = df_clean.copy()  # Avoid modifying a slice.
df_clean["years_exp"] = df_clean["years_exp"].fillna(median_exp)

# Step 5: confirm no missing values remain.
print("Missing after:")
print(df_clean.isna().sum())
print()
print("Rows remaining:", len(df_clean))
df_clean

Missing before:
name          1
department    0
salary        2
years_exp     2
city          0
start_date    0
dtype: int64

Missing after:
name          0
department    0
salary        0
years_exp     0
city          0
start_date    0
dtype: int64

Rows remaining: 9


,name,department,salary,years_exp,city,start_date
0,Valentina Torres,Engineering,4500000,5.0,Bogotá,2019-03-15
1,SANTIAGO GÓMEZ,marketing,3200000,3.0,Medellín,2021-06-01
2,camila restrepo,Engineering,5100000,7.0,Bogotá,2017-08-20
4,Isabella Vargas,Engineering,4800000,4.0,Bogotá,2018-05-07
5,daniel herrera,Marketing,3500000,4.0,Medellín,2020-11-30
6,Sofía López,hr,3100000,1.0,Cali,2023-02-14
9,Julián Pérez,HR,2800000,2.0,Cali,2022-03-05
10,Laura Suárez,Engineering,4200000,5.0,Bogotá,2019-12-01
11,Carlos Rodríguez,Marketing,3700000,4.0,Medellín,2020-04-18


---
## 🔷 Block 2 — Type Conversion and String Cleaning

### Analogy: washing vegetables before cooking

When you buy vegetables at the market, they come with dirt, uneven sizes, and sometimes the wrong item mixed in. Before you can cook, you have to wash, trim, and sort them.

String columns are the same. Real-world data comes with:
- Extra spaces: `"  Bogotá"` instead of `"Bogotá"`
- Mixed case: `"Engineering"`, `"engineering"`, `"ENGINEERING"` — three values that should be one
- Numbers stored as text: `"3200000"` instead of `3200000`

Pandas `.str` methods are your kitchen tools for cleaning text.

### The type problem: salary as text

Look at what `.info()` told us: `salary` has dtype `object`. That means Pandas is treating it as text, not as a number. You cannot calculate an average of text.

In [13]:
# Check the dtype of salary.
print("dtype of salary:", df_clean["salary"].dtype)
print()
# Try to compute the mean — this will fail.
# df_clean["salary"].mean()   # Uncomment to see the error.

dtype of salary: object



In [14]:
# pd.to_numeric() converts a column to a number.
# errors='coerce' means: if a value cannot be converted, replace it with NaN.
df_clean = df_clean.copy()
df_clean["salary"] = pd.to_numeric(df_clean["salary"], errors="coerce")

print("dtype of salary after conversion:", df_clean["salary"].dtype)
print("Mean salary:", df_clean["salary"].mean())

dtype of salary after conversion: int64
Mean salary: 3877777.777777778


### Converting text to dates

In [15]:
# pd.to_datetime() converts a string column to proper date values.
df_clean["start_date"] = pd.to_datetime(df_clean["start_date"])

print("dtype of start_date after conversion:", df_clean["start_date"].dtype)
print(df_clean["start_date"].head())

dtype of start_date after conversion: datetime64[us]
0   2019-03-15
1   2021-06-01
2   2017-08-20
4   2018-05-07
5   2020-11-30
Name: start_date, dtype: datetime64[us]


In [16]:
# Once a column is datetime, you can extract parts with .dt.
df_clean["start_year"] = df_clean["start_date"].dt.year
df_clean[["name", "start_date", "start_year"]].head(6)

,name,start_date,start_year
0,Valentina Torres,2019-03-15,2019
1,SANTIAGO GÓMEZ,2021-06-01,2021
2,camila restrepo,2017-08-20,2017
4,Isabella Vargas,2018-05-07,2018
5,daniel herrera,2020-11-30,2020
6,Sofía López,2023-02-14,2023


### String cleaning with `.str` methods

All Pandas string methods live under `.str`. They apply the operation to **every value** in the column at once — no loop needed.

| Method | What it does | Example |
|--------|--------------|---------|
| `.str.strip()` | Remove leading/trailing spaces | `" Bogotá "` → `"Bogotá"` |
| `.str.lower()` | Lowercase everything | `"MARKETING"` → `"marketing"` |
| `.str.upper()` | Uppercase everything | `"bogotá"` → `"BOGOTÁ"` |
| `.str.title()` | Title Case | `"camila restrepo"` → `"Camila Restrepo"` |
| `.str.replace("a", "b")` | Replace text | `"Bogotá"` → `"Bogota"` |
| `.str.contains("pattern")` | Boolean: does it contain this? | `"Engineering"` → `True` |

In [17]:
# Look at the city column — some values have extra spaces.
print("Before strip:")
print(df_clean["city"].tolist())

df_clean["city"] = df_clean["city"].str.strip()

print("\nAfter strip:")
print(df_clean["city"].tolist())

Before strip:
['Bogotá', 'Medellín', 'Bogotá', ' Bogotá', 'Medellín', 'Cali', 'Cali', 'Bogotá', 'Medellín']

After strip:
['Bogotá', 'Medellín', 'Bogotá', 'Bogotá', 'Medellín', 'Cali', 'Cali', 'Bogotá', 'Medellín']


In [21]:
# Some city values have inconsistent case: 'bogotá' vs 'Bogotá'.
# Normalize: lowercase first, then Title Case.
df_clean["city"] = df_clean["city"].str.lower().str.title()

#df_clean["city"] = df_clean["city"].str.title().str.lower().str.upper()

print("City values after normalization:")
print(df_clean["city"].value_counts())

City values after normalization:
city
Bogotá      4
Medellín    3
Cali        2
Name: count, dtype: int64


In [22]:
# Look at the department column — counting unique values reveals the mess.
print("Department value counts BEFORE cleaning:")
print(df_clean["department"].value_counts())

Department value counts BEFORE cleaning:
department
Engineering    4
Marketing      2
marketing      1
hr             1
HR             1
Name: count, dtype: int64


**The problem:** Pandas treats `"Marketing"`, `"marketing"`, and `"MARKETING"` as three completely different values. If we group by department, we get six rows instead of three.

The fix: normalize everything to the same case before analyzing.

In [23]:
# Normalize department: lowercase → Title Case.
df_clean["department"] = df_clean["department"].str.lower().str.title()

print("Department value counts AFTER cleaning:")
print(df_clean["department"].value_counts())

Department value counts AFTER cleaning:
department
Engineering    4
Marketing      3
Hr             2
Name: count, dtype: int64


In [24]:
# Normalize names: strip spaces, then Title Case.
df_clean["name"] = df_clean["name"].str.strip().str.title()

print("Names after normalization:")
print(df_clean["name"].tolist())

Names after normalization:
['Valentina Torres', 'Santiago Gómez', 'Camila Restrepo', 'Isabella Vargas', 'Daniel Herrera', 'Sofía López', 'Julián Pérez', 'Laura Suárez', 'Carlos Rodríguez']


In [28]:
# .str.contains() returns a boolean Series — useful for filtering.
# Find all employees whose name contains 'López'.
mask = df_clean["name"].str.contains("López", na=False)
df_clean[mask]

,name,department,salary,years_exp,city,start_date,start_year
6,Sofía López,Hr,3100000,1.0,Cali,2023-02-14,2023


---
### Exercise 2

**🧠 Before you code — apply the 5 steps:**

- **Input:** `df_raw` — the original messy DataFrame (with all the original problems).
- **Output:** A fully cleaned DataFrame where:
  - `salary` is numeric (not text).
  - `department` is standardized (Title Case, no inconsistencies).
  - `city` has no leading/trailing spaces and is Title Case.
  - `name` is Title Case with no extra spaces.

**Decompose:**
1. Start from `df_raw` and make a copy.
2. Convert the salary column from text to number.
3. Standardize department names: lowercase then Title Case.
4. Strip whitespace from city, then normalize case.
5. Strip whitespace from name, then apply Title Case.

**Pseudocode:**
```
Copy the raw DataFrame into a new variable.
Convert salary column to numeric, turning failures into NaN.
Apply lowercase then Title Case to the department column.
Strip spaces from city column, then apply lowercase and Title Case.
Strip spaces from name column, then apply Title Case.
Print value counts for department and city to verify.
```

**Pattern:** Filter (detect and standardize each column in sequence).

In [29]:
# Exercise 2 — Clean the raw dataset from scratch.

# Start from the original raw data.
df_ex2 = df_raw.copy()

# TODO 1: Convert salary to numeric.
df_ex2["salary"] = pd.to_numeric(df_ex2["salary"])

# TODO 2: Standardize department names.
df_ex2["department"] =df_ex2["department"].str.lower().str.title()

# TODO 3: Fix city — strip spaces then normalize case.
df_ex2["city"] =df_ex2["city"].str.strip().str.title()

# TODO 4: Fix name — strip spaces then Title Case.
df_ex2["name"] =df_ex2["name"].str.strip().str.title()

# Verify.
print("Department counts:", df_ex2["department"].value_counts().to_dict())
print("City counts:", df_ex2["city"].value_counts().to_dict())

Department counts: {'Engineering': 5, 'Marketing': 4, 'Hr': 3}
City counts: {'Bogotá': 5, 'Medellín': 4, 'Cali': 3}


---
## 🔷 Block 3 — groupby and Aggregation (In Depth)

### Bridge from Module 4

In Module 4 you wrote this function by hand:

```python
def group_by(records, key):
    groups = {}
    for record in records:
        value = record[key]
        if value not in groups:
            groups[value] = []
        groups[value].append(record)
    return groups
```

You then looped over the groups to compute averages manually — which took another 5–10 lines.

Now with Pandas:

```python
df.groupby("department")["salary"].mean()
```

**One line. Same result.** You built it by hand in M4 precisely so you know what is happening inside — Pandas is not magic, it is just doing the same loop automatically.

### What groupby does — step by step

```
df.groupby("department")["salary"].mean()
```

Pandas does three things:
1. **Split** — divide the DataFrame into groups, one per unique value in `department`.
2. **Apply** — run `.mean()` on the `salary` column of each group separately.
3. **Combine** — put the results back together into a single Series.

This is called the **Split-Apply-Combine** pattern.

For Block 3, we use `df_clean` — the cleaned version from Exercise 1 plus Block 2. Let's confirm it is ready.

In [30]:
# Confirm df_clean is ready for analysis.
print("Shape:", df_clean.shape)
print("dtypes:")
print(df_clean.dtypes)
print()
print("Missing values:")
print(df_clean.isna().sum())

Shape: (9, 7)
dtypes:
name                     str
department               str
salary                 int64
years_exp            float64
city                     str
start_date    datetime64[us]
start_year             int32
dtype: object

Missing values:
name          0
department    0
salary        0
years_exp     0
city          0
start_date    0
start_year    0
dtype: int64


In [31]:
# Average salary per department.
df_clean.groupby("department")["salary"].mean()

department
Engineering    4.650000e+06
Hr             2.950000e+06
Marketing      3.466667e+06
Name: salary, dtype: float64

In [32]:
# Total salary cost per department.
df_clean.groupby("department")["salary"].sum()

department
Engineering    18600000
Hr              5900000
Marketing      10400000
Name: salary, dtype: int64

In [33]:
# Number of employees per department.
df_clean.groupby("department")["name"].count()

department
Engineering    4
Hr             2
Marketing      3
Name: name, dtype: int64

In [34]:
# Min and max salary per department.
print("Min salary per department:")
print(df_clean.groupby("department")["salary"].min())
print()
print("Max salary per department:")
print(df_clean.groupby("department")["salary"].max())

Min salary per department:
department
Engineering    4200000
Hr             2800000
Marketing      3200000
Name: salary, dtype: int64

Max salary per department:
department
Engineering    5100000
Hr             3100000
Marketing      3700000
Name: salary, dtype: int64


### `.agg()` — multiple statistics at once

Instead of writing four separate groupby calls, `.agg()` lets you compute multiple statistics in a single step.

In [35]:
# Compute mean, max, and min of salary AND mean of years_exp — all at once.
dept_summary = df_clean.groupby("department").agg(
    salary_mean=("salary", "mean"),
    salary_max=("salary", "max"),
    salary_min=("salary", "min"),
    exp_mean=("years_exp", "mean")
)
dept_summary

,salary_mean,salary_max,salary_min,exp_mean
department,,,,
Engineering,4.650000e+06,5100000,4200000,5.250000
Hr,2.950000e+06,3100000,2800000,1.500000
Marketing,3.466667e+06,3700000,3200000,3.666667


### `.reset_index()` — turn the result back into a regular DataFrame

After a groupby, the group column becomes the **index** (the row label). This is often inconvenient. `.reset_index()` moves it back to a regular column.

In [ ]:
# Without reset_index: department is the index.
print("Without reset_index:")
result = df_clean.groupby("department")["salary"].mean()
print(type(result))   # This is a Series, not a DataFrame.
print(result)
print()

# With reset_index: department becomes a regular column again.
print("With reset_index:")
result_df = df_clean.groupby("department")["salary"].mean().reset_index()
print(type(result_df))   # This is a DataFrame.
print(result_df)

### Grouping by multiple columns

Pass a list of column names to group by two or more categories at the same time.

In [ ]:
# Count employees per department AND city.
df_clean.groupby(["department", "city"])["name"].count().reset_index()

---
### Exercise 3

**🧠 Before you code — apply the 5 steps:**

- **Input:** `df_clean` — the cleaned employees DataFrame.
- **Output:** Four summary tables answering four business questions.

**Decompose:**
1. Compute the average salary for each department.
2. Count the number of employees in each city.
3. Find the maximum years_exp in each department.
4. Count the number of employees per department AND city combined.

**Pseudocode:**
```
Group by department, compute mean of salary → print.
Group by city, count name → print.
Group by department, compute max of years_exp → print.
Group by department and city together, count name → print.
```

**Pattern:** Accumulator (groupby applies an accumulation function to each group).

In [ ]:
# Exercise 3 — Four groupby questions.

# Q1: average salary per department.
print("Q1 — Average salary per department:")
# TODO: write the groupby call.
pass

# Q2: number of employees per city.
print("\nQ2 — Employee count per city:")
# TODO: write the groupby call.
pass

# Q3: max years_exp per department.
print("\nQ3 — Max years of experience per department:")
# TODO: write the groupby call.
pass

# Q4: employees per department AND city.
print("\nQ4 — Employees per department and city:")
# TODO: group by both department and city.
pass

---
## 🔷 Block 4 — Mini-project: End-to-End Cleaning Pipeline

### Scenario

You work as a junior data analyst at a company in Bogotá. Your manager sends you `df_raw` and says:

> “I need you to clean this dataset and then answer three questions for the executive report. I need it by end of day.”

Your questions:
- **Q1:** Which department has the highest average salary?
- **Q2:** How many employees started each year?
- **Q3:** List employees in Bogotá with more than 4 years of experience.

You do not know in advance which rows are dirty. You have to audit first, then fix, then answer.

### Full decomposition — plain English, no code yet

**STEP 1 — Audit the data:**
Look at the first few rows, check dtypes, count missing values per column.

**STEP 2 — Fix types:**
Convert salary from text to number.
Convert start_date from text to date.

**STEP 3 — Fix strings:**
Standardize department names (all same case).
Standardize city names (strip spaces, same case).
Standardize employee names (Title Case).

**STEP 4 — Handle missing values:**
Drop rows where name is missing (cannot identify the employee).
Fill missing salary with the department average (more accurate than overall average).
Fill missing years_exp with the overall median.

**STEP 5 — Answer business questions:**
Q1: Which department has the highest average salary?
Q2: How many employees started each year?
Q3: List employees in Bogotá with more than 4 years of experience.

In [ ]:
# STEP 1 — Audit the data.
print("=== First rows ===")
print(df_raw.head())
print()
print("=== dtypes ===")
print(df_raw.dtypes)
print()
print("=== Missing values per column ===")
print(df_raw.isna().sum())

In [ ]:
# STEP 2 — Fix types.
df_pipeline = df_raw.copy()

# Convert salary from object to numeric.
df_pipeline["salary"] = pd.to_numeric(df_pipeline["salary"], errors="coerce")

# Convert start_date from string to datetime.
df_pipeline["start_date"] = pd.to_datetime(df_pipeline["start_date"])

# Extract start year for Q2.
df_pipeline["start_year"] = df_pipeline["start_date"].dt.year

print("dtypes after type conversion:")
print(df_pipeline.dtypes)

In [ ]:
# STEP 3 — Fix strings.

# Standardize department: lowercase → Title Case.
df_pipeline["department"] = df_pipeline["department"].str.lower().str.title()

# Standardize city: strip spaces → lowercase → Title Case.
df_pipeline["city"] = df_pipeline["city"].str.strip().str.lower().str.title()

# Standardize name: strip spaces → Title Case.
df_pipeline["name"] = df_pipeline["name"].str.strip().str.title()

print("Department counts after fix:")
print(df_pipeline["department"].value_counts())
print()
print("City counts after fix:")
print(df_pipeline["city"].value_counts())

In [ ]:
# STEP 4 — Handle missing values.

# Drop rows where name is missing.
df_pipeline = df_pipeline.dropna(subset=["name"])

# Fill missing salary with the department average.
# We use transform() so the result aligns back to the original rows.
dept_avg_salary = df_pipeline.groupby("department")["salary"].transform("mean")
df_pipeline["salary"] = df_pipeline["salary"].fillna(dept_avg_salary)

# Fill missing years_exp with the overall median.
median_exp = df_pipeline["years_exp"].median()
df_pipeline["years_exp"] = df_pipeline["years_exp"].fillna(median_exp)

print("Missing values after handling:")
print(df_pipeline.isna().sum())
print()
print("Rows remaining:", len(df_pipeline))

In [ ]:
# STEP 5 — Business questions.

# Q1: Which department has the highest average salary?
avg_salary_by_dept = df_pipeline.groupby("department")["salary"].mean().reset_index()
avg_salary_by_dept.columns = ["department", "avg_salary"]
avg_salary_by_dept = avg_salary_by_dept.sort_values("avg_salary", ascending=False)

print("Q1 — Average salary by department:")
print(avg_salary_by_dept)
print()
best_dept = avg_salary_by_dept.iloc[0]["department"]
best_salary = avg_salary_by_dept.iloc[0]["avg_salary"]
print(f"Answer: {best_dept} has the highest average salary: ${best_salary:,.0f} COP")

In [ ]:
# Q2: How many employees started each year?
employees_per_year = df_pipeline.groupby("start_year")["name"].count().reset_index()
employees_per_year.columns = ["year", "employee_count"]
employees_per_year = employees_per_year.sort_values("year")

print("Q2 — Employees by start year:")
print(employees_per_year)

In [ ]:
# Q3: List employees in Bogotá with more than 4 years of experience.
# Pattern: Filter → two conditions combined with &.
mask_bogota = df_pipeline["city"] == "Bogotá"
mask_exp = df_pipeline["years_exp"] > 4

bogota_senior = df_pipeline[mask_bogota & mask_exp][["name", "department", "salary", "years_exp"]]

print("Q3 — Senior employees in Bogotá (>4 years exp):")
print(bogota_senior)

---
## 🟣 Optional · pytest — Testing Cleaning Functions

> Skip this section if pytest was not activated for your course.

### Why test cleaning functions?

When you wrap a cleaning step in a function, you can test it automatically. If you later change the function and accidentally break it, your test will catch the problem immediately.

**Rule of thumb:** if a cleaning step is applied to multiple datasets (or will be reused), wrap it in a function and test it.

In [ ]:
def clean_department(df):
    """Normalize the department column to Title Case and drop rows where department is null.

    Parameters
    ----------
    df : pd.DataFrame
        DataFrame with a 'department' column.

    Returns
    -------
    pd.DataFrame
        DataFrame with department column normalized and no null department values.
    """
    df = df.copy()
    df = df.dropna(subset=["department"])
    df["department"] = df["department"].str.lower().str.title()
    return df


# Quick manual test.
sample = pd.DataFrame({"department": ["engineering", "MARKETING", "HR", None]})
result = clean_department(sample)
print(result)

### Writing tests for `clean_department`

Save the following as `test_homework_module7.py` in your `Module7` folder.

Each test asks a specific question:
- Does the function return a DataFrame?
- Are all department values in Title Case?
- Are null department rows removed?

In [ ]:
# Copy this to test_homework_module7.py and run with: pytest test_homework_module7.py -v

# import pandas as pd
# from homework_module7 import clean_department
#
# def test_clean_department_returns_dataframe():
#     df_input = pd.DataFrame({"department": ["engineering", "HR"]})
#     result = clean_department(df_input)
#     assert isinstance(result, pd.DataFrame)
#
# def test_clean_department_normalizes_case():
#     df_input = pd.DataFrame({"department": ["engineering", "MARKETING", "HR"]})
#     result = clean_department(df_input)
#     expected = ["Engineering", "Marketing", "Hr"]
#     assert result["department"].tolist() == expected
#
# def test_clean_department_removes_nulls():
#     df_input = pd.DataFrame({"department": ["Engineering", None, "HR"]})
#     result = clean_department(df_input)
#     assert result["department"].notna().all()
#     assert len(result) == 2

print("Paste the commented code above into test_homework_module7.py")

---
## 📝 Homework — Module 7

### Messy products dataset

You receive the following messy products dataset. Your task: clean it completely, then answer two business questions.

**Tasks:**
1. **Audit** — print dtypes and the count of missing values per column.
2. **Fix strings** — name → Title Case; category → lowercase; strip extra spaces from both.
3. **Fix types** — convert `price` to numeric; use `errors='coerce'`.
4. **Handle missing values:**
   - Fill missing `price` with the median price for that category.
   - Fill missing `stock` with `0`.
   - Fill missing `supplier` with `"Unknown"`.
5. **Business questions with groupby:**
   - **Q1:** What is the total stock value (`price × stock`) per supplier?
   - **Q2:** What is the average price per category?

*Delivery: bring `homework_module7.py` to the next session. It must run without errors.*

In [ ]:
# homework_module7.py — starter code.
# Copy this cell into a new file called homework_module7.py and complete the TODOs.

import pandas as pd
import numpy as np

products_raw = pd.DataFrame({
    "name":     ["laptop", "PHONE", "Tablet", "headphones", "  Monitor  ", "keyboard"],
    "category": ["electronics", "Electronics", "ELECTRONICS", "electronics", "Electronics", "electronics"],
    "price":    ["2500000", "1200000", None, "350000", "900000", "150000"],
    "stock":    [12, None, 35, 200, 15, 50],
    "supplier": ["TechCorp", "TechCorp", "MobilePro", None, "TechCorp", "KeyMaster"]
})

# STEP 1 — Audit.
# TODO: print dtypes and missing value counts.
pass

# STEP 2 — Fix strings.
df = products_raw.copy()
# TODO: name → Title Case (strip spaces first).
# TODO: category → lowercase (strip spaces first).
pass

# STEP 3 — Fix types.
# TODO: convert price to numeric.
pass

# STEP 4 — Handle missing values.
# TODO: fill missing price with the category median.
# TODO: fill missing stock with 0.
# TODO: fill missing supplier with 'Unknown'.
pass

# STEP 5 — Business questions.
# Create a stock_value column first.
# TODO: df['stock_value'] = price * stock.
pass

# Q1: total stock value per supplier.
# TODO: groupby supplier, sum stock_value.
pass

# Q2: average price per category.
# TODO: groupby category, mean price.
pass

---
## 📋 Module 7 Summary

| Concept | Syntax | Key point |
|---------|--------|-----------|
| Count nulls | `df.isna().sum()` | Per column |
| Drop null rows | `df.dropna(subset=["col"])` | Only where col is null |
| Fill nulls | `df.fillna(value)` | Or `.fillna(df["col"].mean())` |
| Convert to number | `pd.to_numeric(col, errors="coerce")` | Invalid → NaN |
| Convert to date | `pd.to_datetime(col)` | Enables `.dt.year` etc. |
| Strip whitespace | `df["col"].str.strip()` | Leading and trailing |
| Normalize case | `df["col"].str.lower()` | Then `.title()` for names |
| Group and aggregate | `df.groupby("col").agg({...})` | Multiple stats at once |
| Reset index | `.reset_index()` | After groupby |

**Next module:** Visualization — communicating data with charts 📈